In [1]:
import math

import torch
from torch import nn
from torch.nn import functional as F
import torchvision.models as models

from torch.profiler import profile, ProfilerActivity, record_function

In [2]:
d, h = 2048, 16
h_d: int = d // h
dropout: float = 0.1

query = nn.Linear(d, d, bias=False).cuda()
key = nn.Linear(d, d, bias=False).cuda()
value = nn.Linear(d, d, bias=False).cuda()
proj = nn.Linear(d, d).cuda()

drop = nn.Dropout(dropout)

In [3]:
def mha(x, query_layer, key_layer, value_layer, proj_layer):
    B = x.shape[0]

    q = query_layer(x)
    k = key_layer(x)
    v = value_layer(x)

    q = q.view(B, -1, h, h_d).transpose(1, 2)
    k = k.view(B, -1, h, h_d).transpose(1, 2)
    v = v.view(B, -1, h, h_d).transpose(1, 2)

    logits = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(h_d)

    attn = F.softmax(logits, dim=-1)
    # attn = drop(attn)

    out = torch.matmul(attn, v)

    out = out.transpose(1, 2).contiguous().view(B, -1, d)

    return proj_layer(out)

x = torch.rand((1, 5000, d)).cuda()

%timeit mha(x, query, key, value, proj)
# mha(x, query, key, value, proj)

40 ms ± 43.2 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [5]:
def mha_2(x, query_layer, key_layer, value_layer, proj_layer):
    B, T = x.shape[:2]

    k = key_layer(x).view(B, T, h, h_d).transpose(1, 2)
    v = value_layer(x).view(B, T, h, h_d).transpose(1, 2)

    out = torch.empty_like(x)
    step = 100
    for i in range(0, 5000, step):
        q = query_layer(x[:, i:i+step])
        q = q.view(B, step, h, h_d).transpose(1, 2)

        logits = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(h_d)

        attn = F.softmax(logits, dim=-1)
        # attn = drop(attn)

        out_ = torch.matmul(attn, v)

        out_ = out_.transpose(1, 2).contiguous().view(B, -1, d)
        out[:, i:i+step] = proj_layer(out_)

    return out

# x = torch.rand((1, 5000, d)).cuda()
# assert (mha(x, query, key, value, proj) == mha_2(x, query, key, value, proj)).all(), "outputs do not match"
%timeit mha_2(x, query, key, value, proj)

69.1 ms ± 181 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [6]:
with profile(activities=[ProfilerActivity.CUDA], profile_memory=True, record_shapes=True) as prof:
    with record_function("model_inference"):
        mha(x, query, key, value, proj)

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                  cudaDeviceSynchronize        99.74%      36.553ms        99.74%      36.553ms      36.553ms       0.000us         0.00%       0.000us       0.000us           0 b           0 b             1  
                                       cudaLaunchKernel         0.15%      54.000us         0.1

STAGE:2026-05-19 15:22:09 530333:530333 ActivityProfilerController.cpp:314] Completed Stage: Warm Up
STAGE:2026-05-19 15:22:09 530333:530333 ActivityProfilerController.cpp:320] Completed Stage: Collection
STAGE:2026-05-19 15:22:09 530333:530333 ActivityProfilerController.cpp:324] Completed Stage: Post Processing
[W collection.cpp:936] Warning: Failed to recover relationship between all profiler and kineto events: 22 vs. 0  reassociated. (function reassociate)


In [7]:
with profile(activities=[ProfilerActivity.CUDA], profile_memory=True, record_shapes=True) as prof:
    with record_function("model_inference"):
        mha_2(x, query, key, value, proj)

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                  cudaDeviceSynchronize        95.03%      52.632ms        95.03%      52.632ms      52.632ms       0.000us         0.00%       0.000us       0.000us           0 b           0 b             1  
                                       cudaLaunchKernel         3.58%       1.984ms         3.5

STAGE:2026-05-19 15:22:10 530333:530333 ActivityProfilerController.cpp:314] Completed Stage: Warm Up
STAGE:2026-05-19 15:22:10 530333:530333 ActivityProfilerController.cpp:320] Completed Stage: Collection
STAGE:2026-05-19 15:22:10 530333:530333 ActivityProfilerController.cpp:324] Completed Stage: Post Processing
[W collection.cpp:936] Warning: Failed to recover relationship between all profiler and kineto events: 906 vs. 0  reassociated. (function reassociate)


In [23]:
prof.export_chrome_trace("trace.json")

In [18]:
model = models.resnet18()
inputs = torch.randn(5, 3, 224, 224)

In [19]:
with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    with record_function("model_inference"):
        model(inputs)

STAGE:2026-05-19 13:45:39 527137:527137 ActivityProfilerController.cpp:314] Completed Stage: Warm Up
STAGE:2026-05-19 13:45:39 527137:527137 ActivityProfilerController.cpp:320] Completed Stage: Collection
STAGE:2026-05-19 13:45:39 527137:527137 ActivityProfilerController.cpp:324] Completed Stage: Post Processing


In [20]:
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))

---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                  model_inference        -0.49%    -379.000us       100.00%      78.013ms      78.013ms             1  
                     aten::conv2d         3.18%       2.481ms        78.49%      61.230ms       3.062ms            20  
                aten::convolution         0.29%     228.000us        78.37%      61.139ms       3.057ms            20  
               aten::_convolution         0.18%     141.000us        78.08%      60.911ms       3.046ms            20  
         aten::mkldnn_convolution        77.68%      60.602ms        77.90%      60.770ms       3.038ms            20  
                 aten::batch_norm       